## SAX, Hierarchical clustering and volatility prediction

This notebook keeps the data-specific loading, filtering, interpolation, denoising, detrending, and robust normalization logic unchanged in spirit, while replacing the analysis layer with:

1. official `tslearn` SAX representations,
2. SAX MINDIST distance on symbolic strings,
3. agglomerative hierarchical clustering with complete linkage,
4. subsample-based clustering stability by `k`,
5. explainable cluster-level Google Trends indices

Outputs are prioritized as CSV files. Only a small number of diagnostic figures are saved.

## 0. Configuration

Edit paths and headline modeling choices here. The rest of the notebook is designed to run top-to-bottom.

In [25]:
from __future__ import annotations

import json
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import savgol_filter
from scipy.spatial.distance import squareform, pdist
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from tslearn.piecewise import SymbolicAggregateApproximation, PiecewiseAggregateApproximation
from arch import arch_model

# -----------------------------------------------------------------------------
# PATHS -- keep folder structure consistent with the current project
# -----------------------------------------------------------------------------
DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_13")

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
DJ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\DOWJONES_data.csv")
NASDAQ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\NASDAQ100_data.csv")
RUSSELL_PATH = Path(r"C:\Python\CSUREMM\shock_detection\RUSSELL2000_data.csv")

STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02*.csv"

START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# -----------------------------------------------------------------------------
# Data-specific filtering and preprocessing -- preserved from the current setup
# -----------------------------------------------------------------------------
MAX_ZERO_SHARE = 0.50
MAX_MISSING_SHARE = 0.02
MAX_INTERPOLATE_GAP = 7

DENOISE_WINDOW = 9
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91

# -----------------------------------------------------------------------------
# SAX and hierarchical clustering
# -----------------------------------------------------------------------------
MINDIST_CHUNK_SIZE = 256
K_RANGE = range(2, 13)
RANDOM_STATE = 42
N_SEGMENTS = 96
ALPHABET_SIZE = 9
FINAL_K = 6                        # manual fallback; overridden if AUTO_SELECT_K is True

# --- Fix: breakpoints were silently assumed Gaussian even though the pooled PAA
# coefficients turn out to be visibly non-Gaussian (see the fit diagnostic in
# Section 2). "empirical" derives breakpoints from the observed distribution
# instead of theoretical N(0,1) quantiles. Set back to "gaussian" to reproduce
# the original textbook-SAX behavior.
BREAKPOINT_MODE = "empirical"       # {"gaussian", "empirical"}

# --- Fix: N_SEGMENTS / ALPHABET_SIZE were hardcoded with no check of how much
# geometry they preserve. This grid is used purely as a diagnostic ablation
# (Section 3) that reports how tight MINDIST is against true Euclidean distance
# at each (w, a) combination, so the hardcoded choice above can be justified
# or revisited against evidence rather than left arbitrary.
SEGMENT_GRID = [48, 64, 96, 128]
ALPHABET_GRID = [5, 7, 9, 11]
ABLATION_SAMPLE_SIZE = 150          # terms subsampled for the (w, a) tightness check

# --- Fix: consistency + scale invariance + richness (Kleinberg 2002) only admit
# a non-trivial clustering function at the *dendrogram* level under single
# linkage (Zadeh & Ben-David 2009); complete linkage does not have this
# guarantee. Default to single linkage, but empirically compare all four
# candidates via the same stability + silhouette machinery used for k
# (Section 4) so the choice is justified rather than assumed either way.
LINKAGE_METHOD = "ward"           # {"single", "average", "complete", "ward"}
LINKAGE_CANDIDATES = ["single", "average", "complete", "ward"]

# --- Fix: MINDIST produces exact/near-exact ties for genuinely distinct series
# (a direct consequence of quantization). Within TIE_EPS of each other, merge
# order is broken using the true Euclidean distance on the underlying series
# instead of being left to array/index order inside scipy's linkage.
TIE_EPS = 1e-6

# --- Fix: FINAL_K was hardcoded at 6 even though the pipeline's own stability
# + silhouette sweep (Section 4) ranked k in {10, 11, 12} higher. When True,
# FINAL_K is overridden by the best k (by stability, then silhouette) subject
# to a minimum cluster size floor, and the manual value above is only kept as
# a logged comparison point. Set to False to force the manual FINAL_K.
AUTO_SELECT_K = True
MIN_ACCEPTABLE_CLUSTER_SIZE = 15

# Stability: each iteration draws two overlapping subsamples and compares labels
STABILITY_SUBSAMPLES = 80
STABILITY_FRACTION = 0.80

# Cluster-index construction
# --- Fix: this used to be redefined downstream with a different value (0.50
# here vs. 0.75 in the Section 5 cell); single source of truth now.
CORE_STABILITY_QUANTILE = 0.75
MIN_CORE_TERMS = 10
TOP_N_CORE_TERMS = 10

# Prediction
TARGET_HORIZONS = [1, 5]
MIN_TRAIN = 500
RIDGE_ALPHAS = np.logspace(-4, 4, 25)

SUBDIRS = [
    "00_provenance",
    "01_preprocessed_panel",
    "02_sax",
    "03_hierarchical_clustering",
    "04_stability",
    "05_cluster_indices",
    "06_validity_checks",
]

for sub in SUBDIRS:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: C:\Python\CSUREMM\output\sax_tests_july_13


## 1. Load, filter, denoise, detrend, and normalize

In [2]:
def clean_term_from_folder(folder: Path) -> str:
    return folder.name.replace("_", " ").strip()


def find_value_column(df: pd.DataFrame) -> str:
    date_like = {"time", "date", "week"}
    candidates = [c for c in df.columns if c.lower() not in date_like]

    if not candidates:
        raise ValueError("No value column found.")

    numeric = [
        c for c in candidates
        if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors="coerce"))
    ]

    return numeric[0] if numeric else candidates[0]


def max_consecutive_missing(s: pd.Series) -> int:
    missing = s.isna()

    if not missing.any():
        return 0

    groups = (missing != missing.shift()).cumsum()

    return int(
        missing
        .groupby(groups)
        .sum()
        .max()
    )


def load_one_series(folder: Path) -> pd.Series:
    stitched_dir = folder / STITCHED_SUBDIR
    files = sorted(stitched_dir.glob(STITCHED_GLOB)) if stitched_dir.exists() else []

    if not files:
        raise FileNotFoundError("missing stitched file")

    path = files[-1]
    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"time", "date", "week"}),
        df.columns[0],
    )

    value_col = find_value_column(df)

    dates = pd.to_datetime(df[date_col], errors="coerce")
    values = pd.to_numeric(df[value_col], errors="coerce")

    s = pd.Series(
        values.values,
        index=dates,
        name=clean_term_from_folder(folder),
    )

    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")]
    s = s.sort_index()

    s = s.loc[START_DATE:END_DATE]

    full_index = pd.date_range(
        START_DATE,
        END_DATE,
        freq="D",
    )

    return s.reindex(full_index)


def build_panel(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    series = {}

    folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])

    expected_days = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    for folder in folders:
        term = clean_term_from_folder(folder)

        try:
            s = load_one_series(folder)

            observed_share = float(s.notna().mean())
            missing_share = float(s.isna().mean())
            zero_share = float((s.dropna() == 0).mean())
            longest_missing_gap = max_consecutive_missing(s)
            n_unique = int(s.nunique(dropna=True))

            reason = "kept"

            if zero_share > MAX_ZERO_SHARE:
                reason = "high_zero_share"
            elif missing_share > MAX_MISSING_SHARE:
                reason = "high_missing_share"
            elif longest_missing_gap > MAX_INTERPOLATE_GAP:
                reason = "long_missing_gap"
            elif n_unique <= 2:
                reason = "near_constant"

            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": int(s.notna().sum()),
                "observed_share": observed_share,
                "missing_share": missing_share,
                "zero_share": zero_share,
                "longest_missing_gap": longest_missing_gap,
                "n_unique": n_unique,
                "drop_reason": reason,
            })

            if reason == "kept":
                series[term] = s

        except Exception as e:
            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": 0,
                "observed_share": 0.0,
                "missing_share": 1.0,
                "zero_share": np.nan,
                "longest_missing_gap": np.nan,
                "n_unique": 0,
                "drop_reason": str(e),
            })

    panel = pd.DataFrame(series).sort_index()
    funnel = pd.DataFrame(rows).sort_values(["drop_reason", "term"])

    return panel, funnel


def interpolate_small_gaps(s: pd.Series) -> pd.Series:
    return (
        s.astype(float)
        .interpolate(
            method="time",
            limit=MAX_INTERPOLATE_GAP,
            limit_direction="both",
        )
    )


def denoise_series(s: pd.Series) -> pd.Series:
    x = s.astype(float)
    valid = x.dropna()

    if len(valid) < DENOISE_WINDOW or DENOISE_WINDOW % 2 == 0:
        return x

    filled = x.interpolate(
        method="time",
        limit_direction="both",
    )

    return pd.Series(
        savgol_filter(
            filled,
            DENOISE_WINDOW,
            DENOISE_POLYORDER,
        ),
        index=x.index,
        name=x.name,
    )


def detrend_series(s: pd.Series) -> pd.Series:
    trend = s.rolling(
        DETREND_WINDOW,
        center=True,
        min_periods=max(14, DETREND_WINDOW // 4),
    ).median()

    return s - trend


def robust_zscore(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    x = s.astype(float)

    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)

        if pd.isna(std) or std == 0:
            return pd.Series(
                np.zeros(len(x)),
                index=x.index,
                name=x.name,
            )

        return (
            (x - x.mean(skipna=True)) / std
        ).rename(x.name)

    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    mad_ratio = (
        mad / wide_spread
        if wide_spread > 0
        else 1.0
    )

    is_degenerate = mad_ratio < mad_ratio_threshold

    if not is_degenerate:
        return (
            0.6745 * (x - med) / mad
        ).rename(x.name)

    mad_floored = (
        max(mad, mad_floor_frac * wide_spread)
        if wide_spread > 0
        else mad
    )

    z = (
        0.6745 * (x - med) / mad_floored
    ).rename(x.name)

    return z.clip(
        lower=-clip_mad_multiple,
        upper=clip_mad_multiple,
    )


def preprocess_panel(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:

    stages = {}

    stages["filled"] = panel.apply(
        interpolate_small_gaps,
        axis=0,
    )

    stages["denoised"] = stages["filled"].apply(
        denoise_series,
        axis=0,
    )

    stages["detrended"] = stages["denoised"].apply(
        detrend_series,
        axis=0,
    )

    stages["normalized"] = stages["detrended"].apply(
        robust_zscore,
        axis=0,
    )

    return stages["normalized"], stages


def save_panel_stages(panel_raw: pd.DataFrame, stages: dict[str, pd.DataFrame]) -> None:
    panel_raw.to_csv(OUTPUT_DIR / "01_preprocessed_panel" / "panel_raw_retained.csv")
    for name, df in stages.items():
        df.to_csv(OUTPUT_DIR / "01_preprocessed_panel" / f"panel_{name}.csv")

In [3]:
panel_raw, filtering_funnel = build_panel(DATA_DIR)
filtering_funnel.to_csv(OUTPUT_DIR / "00_provenance" / "filtering_funnel.csv", index=False)

panel_norm, stages = preprocess_panel(panel_raw)
panel_norm = panel_norm.dropna(axis=1, how="all")
panel_norm = panel_norm.loc[:, panel_norm.notna().mean() > 0.95]
stages["normalized"] = panel_norm

save_panel_stages(panel_raw, stages)

summary = pd.DataFrame({
    "n_days": [panel_norm.shape[0]],
    "n_terms_retained": [panel_norm.shape[1]],
    "start_date": [panel_norm.index.min()],
    "end_date": [panel_norm.index.max()],
})
summary.to_csv(OUTPUT_DIR / "00_provenance" / "analysis_sample_summary.csv", index=False)

print(summary.to_string(index=False))
print("\nFiltering funnel:")
print(filtering_funnel["drop_reason"].value_counts(dropna=False).to_string())

 n_days  n_terms_retained start_date   end_date
   1612               847 2022-01-01 2026-05-31

Filtering funnel:
drop_reason
kept                  847
high_zero_share       352
high_missing_share      4


In [5]:
panel_norm.isna().sum().sum() == 0

np.True_

## 2. `tslearn` SAX representation

Each retained term is represented as one SAX string. The notebook uses `tslearn.piecewise.SymbolicAggregateApproximation`. Missing values after preprocessing are linearly filled only for the representation step.

In [4]:
def panel_to_tslearn_array(panel: pd.DataFrame) -> np.ndarray:
    filled = panel.astype(float).interpolate("time").ffill().bfill()
    X = filled.T.to_numpy(dtype=float)
    return X[:, :, None]


def sax_to_strings(codes_2d: np.ndarray, alphabet_size: int, index) -> pd.Series:
    alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"[:alphabet_size]))
    strings = ["".join(alphabet[row]) for row in codes_2d]
    return pd.Series(strings, index=index, name="sax_string")


def sax_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian breakpoints used by standard SAX (theoretical N(0,1) assumption)."""
    return norm.ppf(np.arange(1, alphabet_size) / alphabet_size)


def empirical_breakpoints(paa_values: np.ndarray, alphabet_size: int) -> np.ndarray:
    """
    Data-driven analogue of sax_breakpoints(): equiprobable quantiles of the
    pooled, observed PAA-coefficient distribution rather than theoretical
    N(0,1) quantiles. Use when the Gaussian assumption doesn't fit the data
    (see the fit diagnostic immediately below).
    """
    qs = np.arange(1, alphabet_size) / alphabet_size
    return np.quantile(paa_values, qs)


def discretize_paa(paa_values_2d: np.ndarray, breakpoints: np.ndarray) -> np.ndarray:
    """Assign each PAA coefficient to a symbol index given a set of breakpoints."""
    return np.digitize(paa_values_2d, breakpoints)


# throwing away extreme values to avoid distortion in SAX representation
panel_sax = panel_norm.clip(-5, 5)
X_ts = panel_to_tslearn_array(panel_sax)

paa = PiecewiseAggregateApproximation(n_segments=N_SEGMENTS)
paa_coefs = paa.fit_transform(X_ts)[:, :, 0]          # (n_terms, n_segments)
paa_flat = paa_coefs.ravel()

# BREAKPOINT_MODE (config, Section 0) selects Gaussian vs. empirical breakpoints.
if BREAKPOINT_MODE == "empirical":
    active_breakpoints = empirical_breakpoints(paa_flat, ALPHABET_SIZE)
else:
    active_breakpoints = sax_breakpoints(ALPHABET_SIZE)

sax_code_array = discretize_paa(paa_coefs, active_breakpoints)   # (n_terms, n_segments)
sax_codes = sax_code_array[:, :, None]                            # tslearn-style shape

sax_strings = sax_to_strings(sax_code_array, ALPHABET_SIZE, panel_norm.columns)

sax_df = pd.DataFrame({
    "term": sax_strings.index,
    "sax_string": sax_strings.values,
    "n_segments": N_SEGMENTS,
    "alphabet_size": ALPHABET_SIZE,
    "breakpoint_mode": BREAKPOINT_MODE,
})
sax_df.to_csv(OUTPUT_DIR / "02_sax" / "sax_strings.csv", index=False)

symbol_counts = (
    sax_df.assign(symbol=sax_df["sax_string"].map(list))
    .explode("symbol")
    .groupby(["term", "symbol"])
    .size()
    .unstack(fill_value=0)
)
symbol_counts.to_csv(OUTPUT_DIR / "02_sax" / "sax_symbol_counts.csv")

print("SAX strings:", sax_df.shape, " | breakpoint_mode:", BREAKPOINT_MODE)
sax_df.head()

SAX strings: (847, 5)  | breakpoint_mode: empirical


,term,sax_string,n_segments,alphabet_size,breakpoint_mode
0,2020 election map,gfadaebheagifafbebhicfbbaggbggcbfcbaiiaedbiadb...,96,9,empirical
1,2020 election results,fgbgfcchieihahhcaagihhcbeefehdedecbbiidaahiahf...,96,9,empirical
2,2022 election,bcgihaaiifbaaiiceadigeeeeeeeedgchfbdhdbcghidca...,96,9,empirical
3,2022 election results,edehgfchihgcaiieecdigeeeeeeeedfdfebfgecdfeibbb...,96,9,empirical
4,2024 election,gdcfcbeedahhhbccdbfihbabghiahgggheaagigaaiiaaa...,96,9,empirical


### Justify clipping value selection - 5 is the right approach

In [7]:
for c in [2,3,4,5]:
    pct = np.mean(np.abs(panel_norm.to_numpy()) > c)
    print(c, pct)

2 0.18607492214530338
3 0.1243272856176082
4 0.09299351674718244
5 0.07572705886488877


In [5]:
# --- Diagnostic: does the Gaussian breakpoint assumption actually fit? ---
# Standard SAX breakpoints assume PAA coefficients are N(0,1). This checks that
# assumption directly rather than assuming it; excess kurtosis > 0 or a small
# KS p-value indicates the pooled coefficients are more peaked / non-Gaussian
# than the textbook breakpoints assume, which is exactly what motivates
# BREAKPOINT_MODE = "empirical" above.
from scipy import stats as _stats

standardized = (paa_flat - paa_flat.mean()) / paa_flat.std()
excess_kurtosis = _stats.kurtosis(paa_flat, fisher=True)   # 0 for a true Gaussian
ks_stat, ks_p = _stats.kstest(standardized, "norm")

breakpoint_fit = pd.DataFrame({
    "metric": ["excess_kurtosis", "ks_statistic", "ks_pvalue", "n_paa_coefficients", "breakpoint_mode_used"],
    "value": [excess_kurtosis, ks_stat, ks_p, len(paa_flat), BREAKPOINT_MODE],
})
breakpoint_fit.to_csv(OUTPUT_DIR / "07_validity_checks" / "paa_gaussian_fit_diagnostic.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(paa_flat, bins=60, density=True, alpha=0.7, label="pooled PAA coefficients")
xs = np.linspace(paa_flat.min(), paa_flat.max(), 200)
axes[0].plot(xs, norm.pdf(xs, paa_flat.mean(), paa_flat.std()), lw=2, label="fitted Gaussian")
axes[0].set_title("Pooled PAA coefficients vs Gaussian fit")
axes[0].legend(frameon=False)
_stats.probplot(paa_flat, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot vs Gaussian")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_validity_checks" / "paa_gaussian_fit_diagnostic.png", dpi=200)
plt.close(fig)

print(f"Excess kurtosis: {excess_kurtosis:.3f}  (0 = Gaussian; > 0 = more peaked than Gaussian)")
print(f"KS test vs N(0,1): stat={ks_stat:.4f}, p={ks_p:.2e}")
print(f"-> breakpoint_mode currently in use: {BREAKPOINT_MODE}")

Excess kurtosis: 3.887  (0 = Gaussian; > 0 = more peaked than Gaussian)
KS test vs N(0,1): stat=0.1457, p=0.00e+00
-> breakpoint_mode currently in use: empirical


## 3. SAX MINDIST and hierarchical clustering

The clustering distance is SAX **MINDIST**, computed from the (Gaussian or
empirical, per `BREAKPOINT_MODE`) breakpoints established in Section 2.

Before committing to `N_SEGMENTS` / `ALPHABET_SIZE`, a small ablation below
reports how tight MINDIST is against the true Euclidean distance across a grid
of `(n_segments, alphabet_size)` choices -- this is a diagnostic, not an
automatic override, so the hardcoded values above can be checked against
evidence.

Linkage defaults to **single linkage**: it is the only merge rule for which
Kleinberg-style scale invariance + richness + consistency are jointly
satisfiable once stated over the dendrogram (Zadeh & Ben-David, 2009), which
is the property MINDIST-based clustering is implicitly leaning on. All four
standard linkage methods are compared empirically in Section 4 regardless, so
this default is a starting point, not an assumption baked in silently.

MINDIST also produces exact/near-exact distance ties for genuinely distinct
series (a direct consequence of quantization); these are broken using the true
Euclidean distance on the underlying series rather than left to array order.

In [6]:
def sax_symbol_distance_table(alphabet_size: int, breakpoints: np.ndarray | None = None) -> np.ndarray:
    """
    Pairwise SAX symbol distances used in MINDIST.

    Adjacent symbols have distance 0 because their intervals touch. Non-adjacent
    symbols are separated by the gap between the relevant breakpoints. Accepts an
    explicit breakpoints array so the table reflects whichever BREAKPOINT_MODE
    (Gaussian or empirical) was actually used to build the SAX codes.
    """
    bp = sax_breakpoints(alphabet_size) if breakpoints is None else breakpoints
    table = np.zeros((alphabet_size, alphabet_size), dtype=float)

    for i in range(alphabet_size):
        for j in range(alphabet_size):
            if abs(i - j) <= 1:
                table[i, j] = 0.0
            else:
                lo, hi = sorted((i, j))
                table[i, j] = bp[hi - 1] - bp[lo]
    return table


def mindist_tightness_check(
    panel: pd.DataFrame,
    segment_grid: list[int],
    alphabet_grid: list[int],
    sample_size: int,
    seed: int,
) -> pd.DataFrame:
    """
    On a subsample of terms, compare MINDIST to the true Euclidean distance
    across a grid of (n_segments, alphabet_size). MINDIST should always
    lower-bound the true distance (share_mindist_exceeds_euclid ~ 0 validates
    this); mean_tightness_ratio close to 1 means little geometry is lost at
    that (w, a); ratios well below 1 mean the compression/quantization is
    throwing away a lot of the true signal. This is diagnostic only -- it does
    not silently override N_SEGMENTS / ALPHABET_SIZE above.
    """
    rng = np.random.default_rng(seed)
    cols = panel.columns.to_list()
    sample_cols = rng.choice(cols, size=min(sample_size, len(cols)), replace=False)
    sub = panel[sample_cols]

    X = panel_to_tslearn_array(sub)
    n_timestamps = sub.shape[0]
    flat = X[:, :, 0]
    euclid = squareform(pdist(flat, metric="euclidean"))
    mask = ~np.eye(len(sample_cols), dtype=bool)

    rows = []
    for w in segment_grid:
        paa_w = PiecewiseAggregateApproximation(n_segments=w).fit_transform(X)[:, :, 0]
        for a in alphabet_grid:
            bp = (
                empirical_breakpoints(paa_w.ravel(), a)
                if BREAKPOINT_MODE == "empirical"
                else sax_breakpoints(a)
            )
            codes = discretize_paa(paa_w, bp)
            sd = sax_symbol_distance_table(a, bp)
            scale = np.sqrt(n_timestamps / w)
            per_seg = sd[codes[:, None, :], codes[None, :, :]]
            md = scale * np.sqrt((per_seg ** 2).sum(axis=2))

            ratio = md[mask] / np.clip(euclid[mask], 1e-9, None)
            rows.append({
                "n_segments": w,
                "alphabet_size": a,
                "mean_tightness_ratio": float(np.mean(ratio)),
                "median_tightness_ratio": float(np.median(ratio)),
                "share_mindist_exceeds_euclid": float(np.mean(md[mask] > euclid[mask] + 1e-6)),
            })
    return pd.DataFrame(rows)


tightness_df = mindist_tightness_check(
    panel_sax, SEGMENT_GRID, ALPHABET_GRID, ABLATION_SAMPLE_SIZE, RANDOM_STATE
)
tightness_df.to_csv(OUTPUT_DIR / "07_validity_checks" / "mindist_tightness_by_w_a.csv", index=False)

chosen_row = tightness_df[
    (tightness_df["n_segments"] == N_SEGMENTS) & (tightness_df["alphabet_size"] == ALPHABET_SIZE)
]
print("(w, a) tightness ablation (higher mean_tightness_ratio = less information lost):")
print(tightness_df.sort_values("mean_tightness_ratio", ascending=False).to_string(index=False))
print("\nCurrently configured (N_SEGMENTS, ALPHABET_SIZE) row:")
print(chosen_row.to_string(index=False))

(w, a) tightness ablation (higher mean_tightness_ratio = less information lost):
 n_segments  alphabet_size  mean_tightness_ratio  median_tightness_ratio  share_mindist_exceeds_euclid
        128             11              0.388816                0.391334                           0.0
         96             11              0.360789                0.361860                           0.0
        128              9              0.348101                0.350331                           0.0
         96              9              0.322465                0.323694                           0.0
         64             11              0.303398                0.303838                           0.0
        128              7              0.292205                0.293450                           0.0
         64              9              0.273833                0.274399                           0.0
         96              7              0.273356                0.273721                       

In [7]:
def sax_mindist_matrix(
    sax_codes: np.ndarray,
    terms: list[str],
    n_timestamps: int,
    n_segments: int,
    alphabet_size: int,
    symbol_dist: np.ndarray,
    chunk_size: int = MINDIST_CHUNK_SIZE,
) -> pd.DataFrame:
    """
    Compute SAX MINDIST between all term-level SAX strings.

    Parameters
    ----------
    sax_codes:
        Symbol-code array, shape (n_terms, n_segments, 1).
    terms:
        Term names in the same order as sax_codes.
    n_timestamps:
        Original equal-length series length before SAX compression.
    n_segments:
        Number of SAX/PAA segments.
    alphabet_size:
        Number of SAX symbols.
    symbol_dist:
        Precomputed symbol-to-symbol distance table (reflects the breakpoints
        actually used -- Gaussian or empirical -- rather than assuming Gaussian).
    chunk_size:
        Row chunk size to keep memory use stable.

    Returns
    -------
    pd.DataFrame
        Symmetric MINDIST matrix indexed and columned by term.
    """
    codes = np.asarray(sax_codes[:, :, 0], dtype=np.int16)
    n_terms, actual_segments = codes.shape

    if actual_segments != n_segments:
        raise ValueError(f"Expected {n_segments} SAX segments, got {actual_segments}.")
    if len(terms) != n_terms:
        raise ValueError("Number of terms does not match number of SAX code rows.")

    scale = np.sqrt(n_timestamps / n_segments)
    D = np.zeros((n_terms, n_terms), dtype=np.float32)

    for start in range(0, n_terms, chunk_size):
        stop = min(start + chunk_size, n_terms)
        block = codes[start:stop]
        per_segment = symbol_dist[block[:, None, :], codes[None, :, :]]
        D[start:stop, :] = scale * np.sqrt((per_segment ** 2).sum(axis=2))

    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=terms, columns=terms)


def true_euclidean_distance_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    """True (full-resolution) Euclidean distance, used only to break MINDIST ties."""
    X = panel.T.to_numpy(dtype=float)
    D = squareform(pdist(X, metric="euclidean"))
    return pd.DataFrame(D, index=panel.columns, columns=panel.columns)


def break_mindist_ties(mindist_df: pd.DataFrame, euclid_df: pd.DataFrame, eps: float = TIE_EPS) -> pd.DataFrame:
    """
    MINDIST is a many-to-one lower bound and produces exact/near-exact ties for
    genuinely distinct series (see the duplicate-value counts in the SAX
    characterization diagnostics). Within `eps` those ties are effectively
    arbitrary and would otherwise be resolved by array/index order inside
    scipy's linkage. This adds a rank-preserving nudge from the true Euclidean
    distance, strictly smaller than `eps`, so no non-tied comparison is
    reordered but tied comparisons are resolved using real signal.
    """
    M = mindist_df.to_numpy(dtype=float)
    E = euclid_df.reindex(index=mindist_df.index, columns=mindist_df.columns).to_numpy(dtype=float)

    e_rank = E / (E.max() + 1e-9)          # in [0, 1), preserves Euclidean order
    nudge = eps * e_rank                    # strictly smaller than eps
    M_broken = M + nudge
    np.fill_diagonal(M_broken, 0.0)
    M_broken = 0.5 * (M_broken + M_broken.T)
    return pd.DataFrame(M_broken, index=mindist_df.index, columns=mindist_df.columns)


def hierarchical_labels(distance_df: pd.DataFrame, k: int, method: str | None = None) -> pd.Series:
    method = method or LINKAGE_METHOD
    condensed = squareform(distance_df.to_numpy(dtype=float), checks=False)
    Z = linkage(condensed, method=method)
    labels = fcluster(Z, t=k, criterion="maxclust")
    return pd.Series(labels, index=distance_df.index, name="cluster")


symbol_dist_table = sax_symbol_distance_table(ALPHABET_SIZE, active_breakpoints)

mindist = sax_mindist_matrix(
    sax_codes=sax_codes,
    terms=sax_strings.index.to_list(),
    n_timestamps=panel_norm.shape[0],
    n_segments=N_SEGMENTS,
    alphabet_size=ALPHABET_SIZE,
    symbol_dist=symbol_dist_table,
)
mindist.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "sax_mindist_matrix.csv")

mindist_summary = pd.DataFrame({
    "metric": ["n_terms", "n_segments", "alphabet_size", "min_offdiag", "median_offdiag", "mean_offdiag", "max_offdiag"],
    "value": [
        mindist.shape[0],
        N_SEGMENTS,
        ALPHABET_SIZE,
        np.nanmin(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmedian(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmean(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmax(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
    ],
})
mindist_summary.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "sax_mindist_summary.csv", index=False)

n_exact_ties = int(np.isclose(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy(), 0.0, atol=1e-9).sum())
print(f"Exact-zero off-diagonal MINDIST pairs before tie-breaking: {n_exact_ties}")

euclid_true = true_euclidean_distance_matrix(panel_sax)
mindist_tiebroken = break_mindist_ties(mindist, euclid_true, eps=TIE_EPS)
mindist_tiebroken.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "sax_mindist_matrix_tiebroken.csv")

# All downstream clustering (labels, stability, dendrogram) uses the
# tie-broken matrix; mindist / mindist_summary above stay in original MINDIST
# units for interpretable reporting (the nudge is negligible for those).
labels_final = hierarchical_labels(mindist_tiebroken, FINAL_K)

cluster_assignments = labels_final.rename_axis("term").reset_index()
cluster_assignments.to_csv(
    OUTPUT_DIR / "03_hierarchical_clustering" / "cluster_assignments.csv",
    index=False,
)

cluster_sizes = labels_final.value_counts().sort_index().rename_axis("cluster").reset_index(name="n_terms")
cluster_sizes.to_csv(OUTPUT_DIR / "03_hierarchical_clustering" / "cluster_sizes.csv", index=False)

print(cluster_sizes.to_string(index=False))
mindist_summary

Exact-zero off-diagonal MINDIST pairs before tie-breaking: 16
 cluster  n_terms
       1       90
       2       86
       3       58
       4      152
       5       25
       6      436


,metric,value
0,n_terms,847.000000
1,n_segments,96.000000
2,alphabet_size,9.000000
3,min_offdiag,0.000000
4,median_offdiag,32.555050
5,mean_offdiag,31.949610
6,max_offdiag,54.985077


### Some diagnosis on SAX characterization

In [9]:
print("panel_norm shape:", panel_norm.shape)
print("N_SEGMENTS:", N_SEGMENTS)
print("ALPHABET_SIZE:", ALPHABET_SIZE)
print("FINAL_K:", FINAL_K)
print("NaNs:", panel_norm.isna().sum().sum())
print("Constant cols:", (panel_norm.std(axis=0) == 0).sum())

panel_norm shape: (1612, 847)
N_SEGMENTS: 96
ALPHABET_SIZE: 9
FINAL_K: 6
NaNs: 0
Constant cols: 0


In [8]:
codes = np.asarray(sax_codes[:, :, 0], dtype=int)

# 1. How many unique SAX words?
sax_words = ["".join(map(str, row)) for row in codes]
print("unique SAX words:", pd.Series(sax_words).nunique())
print(pd.Series(sax_words).value_counts().head(20))

# 2. Symbol usage
symbol_counts = pd.Series(codes.ravel()).value_counts().sort_index()
print(symbol_counts)

# 3. Per-term number of unique symbols
unique_symbols_per_term = pd.Series(
    [len(set(row)) for row in codes],
    index=sax_strings.index,
    name="n_unique_symbols"
)
print(unique_symbols_per_term.describe())
print(unique_symbols_per_term.value_counts().sort_index())

# 4. Distance distribution
offdiag = mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy().ravel()
offdiag = offdiag[~np.isnan(offdiag)]
print(pd.Series(offdiag).describe())
print(pd.Series(np.round(offdiag, 6)).value_counts().head(20))

unique SAX words: 843
434765278762088442386444444443535415642354811188478000377058100188803575828583466874443885008812    2
632521443077712231587101678076667400686008800080388000660588080367444444443434633752236744215743    2
007810088210088450187644445453687413724174871065786422671777170367444444444773573721133867203748    2
136206577531215576663704766516773321165766617047465167652112676457150565643575512145666562103746    2
650304174068505141782511066166215210880431803187087200430788171367444444444444444645545786106851    1
561652278487077200687721445473434211883007807577186211371688080178444444444543644881025772808880    1
126870088510088240386444444443627513731267832044376331141477011688410444424162266880147786008850    1
414621352177723140387031588066777701686008800186088101170688080367444444443434634862245743116861    1
047208788641226786770701777316884312277767706008662276552223687357150474625564422346765561105644    1
784164464444467676500707834543754434776884083218844016511047

In [10]:
for k in range(2, 11):
    labels = hierarchical_labels(mindist_tiebroken, k)

    print(
        k,
        labels.value_counts().values
    )

2 [671 176]
3 [613 176  58]
4 [461 176 152  58]
5 [436 176 152  58  25]
6 [436 152  90  86  58  25]
7 [436 152  90  58  46  40  25]
8 [436  90  80  72  58  46  40  25]
9 [301 135  90  80  72  58  46  40  25]
10 [301  90  88  80  72  58  47  46  40  25]


## 4. Cluster-size selection and subsample stability

For each `k`, the notebook records compact validation metrics and a stability distribution. Stability repeatedly clusters two overlapping subsamples, intersects their shared terms, and computes adjusted Rand similarity on the shared labels.

In [11]:
def labels_from_submatrix(full_distance: pd.DataFrame, terms: list[str], k: int, method: str | None = None) -> pd.Series:
    method = method or LINKAGE_METHOD
    sub = full_distance.loc[terms, terms]
    return hierarchical_labels(sub, k, method=method)


def stability_for_k(
    full_distance: pd.DataFrame,
    k: int,
    n_subsamples: int,
    fraction: float,
    seed: int,
    method: str | None = None,
) -> pd.DataFrame:
    method = method or LINKAGE_METHOD
    rng = np.random.default_rng(seed)
    terms = np.array(full_distance.index)
    n_take = max(k + 2, int(round(len(terms) * fraction)))
    rows = []

    for b in range(n_subsamples):
        s1 = rng.choice(terms, size=n_take, replace=False).tolist()
        s2 = rng.choice(terms, size=n_take, replace=False).tolist()
        common = sorted(set(s1).intersection(s2))

        if len(common) < max(3, k):
            rows.append({"k": k, "iteration": b, "n_common": len(common), "ari": np.nan})
            continue

        l1 = labels_from_submatrix(full_distance, s1, k, method=method).loc[common]
        l2 = labels_from_submatrix(full_distance, s2, k, method=method).loc[common]
        rows.append({
            "k": k,
            "iteration": b,
            "n_common": len(common),
            "ari": adjusted_rand_score(l1, l2),
        })
    return pd.DataFrame(rows)


def validation_for_k(full_distance: pd.DataFrame, labels: pd.Series, k: int) -> dict:
    D = full_distance.to_numpy()
    labs = labels.loc[full_distance.index].to_numpy()
    sil = silhouette_score(D, labs, metric="precomputed") if len(np.unique(labs)) > 1 else np.nan
    sizes = labels.value_counts()
    return {
        "k": k,
        "silhouette_mindist": sil,
        "min_cluster_size": int(sizes.min()),
        "max_cluster_size": int(sizes.max()),
        "median_cluster_size": float(sizes.median()),
    }


validation_rows = []
stability_tables = []

for k in K_RANGE:
    labels_k = hierarchical_labels(mindist_tiebroken, k)
    validation_rows.append(validation_for_k(mindist_tiebroken, labels_k, k))
    stability_tables.append(
        stability_for_k(
            mindist_tiebroken,
            k=k,
            n_subsamples=STABILITY_SUBSAMPLES,
            fraction=STABILITY_FRACTION,
            seed=RANDOM_STATE + k,
        )
    )

validation_df = pd.DataFrame(validation_rows)
stability_raw = pd.concat(stability_tables, ignore_index=True)
stability_summary = (
    stability_raw
    .groupby("k")
    .agg(
        stability_mean=("ari", "mean"),
        stability_median=("ari", "median"),
        stability_p10=("ari", lambda x: x.quantile(0.10)),
        stability_p90=("ari", lambda x: x.quantile(0.90)),
        mean_common_terms=("n_common", "mean"),
    )
    .reset_index()
)

cluster_selection = validation_df.merge(stability_summary, on="k", how="left")
cluster_selection.to_csv(OUTPUT_DIR / "04_stability" / "cluster_selection_by_k.csv", index=False)
stability_raw.to_csv(OUTPUT_DIR / "04_stability" / "subsample_stability_raw.csv", index=False)

print(cluster_selection.sort_values(["stability_median", "silhouette_mindist"], ascending=False).to_string(index=False))

# --- Fix: linkage method was hardcoded to "complete" with no empirical check,
# even though the axiomatic grounding referenced for this pipeline (Zadeh &
# Ben-David uniqueness result) specifically singles out single linkage. Compare
# all LINKAGE_CANDIDATES at the currently configured k using the same
# stability + silhouette machinery above, rather than assuming one is right.
_k_for_linkage_check = FINAL_K if FINAL_K is not None else int(
    cluster_selection.sort_values(["stability_median", "silhouette_mindist"], ascending=False).iloc[0]["k"]
)

linkage_rows = []
for method in LINKAGE_CANDIDATES:
    labels_m = hierarchical_labels(mindist_tiebroken, _k_for_linkage_check, method=method)
    val = validation_for_k(mindist_tiebroken, labels_m, _k_for_linkage_check)
    stab = stability_for_k(
        mindist_tiebroken, _k_for_linkage_check, STABILITY_SUBSAMPLES, STABILITY_FRACTION, RANDOM_STATE, method=method
    )
    linkage_rows.append({
        "linkage_method": method,
        "k_used": _k_for_linkage_check,
        "silhouette_mindist": val["silhouette_mindist"],
        "min_cluster_size": val["min_cluster_size"],
        "max_cluster_size": val["max_cluster_size"],
        "stability_mean": stab["ari"].mean(),
        "stability_median": stab["ari"].median(),
    })

linkage_comparison = pd.DataFrame(linkage_rows)
linkage_comparison.to_csv(OUTPUT_DIR / "07_validity_checks" / "linkage_method_comparison.csv", index=False)
print(f"\nLinkage method comparison at k={_k_for_linkage_check} (currently configured LINKAGE_METHOD = '{LINKAGE_METHOD}'):")
print(linkage_comparison.sort_values("stability_median", ascending=False).to_string(index=False))

# --- Fix: FINAL_K was hardcoded at 6 even though this exact table ranks
# k in {10, 11, 12} higher on both stability and silhouette. If AUTO_SELECT_K
# is True, pick the best k (by stability, then silhouette) subject to a
# minimum cluster size floor; otherwise keep the manual FINAL_K but still log
# the data-driven choice for comparison so the discrepancy is visible rather
# than silently dropped.
def select_k_from_stability(cluster_selection: pd.DataFrame, min_cluster_size: int) -> int:
    eligible = cluster_selection[cluster_selection["min_cluster_size"] >= min_cluster_size]
    if eligible.empty:
        eligible = cluster_selection
    best = eligible.sort_values(["stability_median", "silhouette_mindist"], ascending=False).iloc[0]
    return int(best["k"])


auto_k = select_k_from_stability(cluster_selection, MIN_ACCEPTABLE_CLUSTER_SIZE)
manual_k = FINAL_K
EFFECTIVE_K = auto_k if AUTO_SELECT_K else manual_k

k_selection_log = pd.DataFrame({
    "metric": ["auto_selected_k", "manual_final_k", "min_acceptable_cluster_size", "auto_select_k_enabled", "effective_k_used"],
    "value": [auto_k, manual_k, MIN_ACCEPTABLE_CLUSTER_SIZE, AUTO_SELECT_K, EFFECTIVE_K],
})
k_selection_log.to_csv(OUTPUT_DIR / "07_validity_checks" / "k_selection_log.csv", index=False)

FINAL_K = EFFECTIVE_K  # everything downstream (filenames included) uses this resolved value
print(f"\nData-driven k (stability + silhouette, min cluster size >= {MIN_ACCEPTABLE_CLUSTER_SIZE}): {auto_k}")
print(f"Manually configured FINAL_K: {manual_k}   ->   AUTO_SELECT_K={AUTO_SELECT_K}   ->   effective k: {FINAL_K}")

 k  silhouette_mindist  min_cluster_size  max_cluster_size  median_cluster_size  stability_mean  stability_median  stability_p10  stability_p90  mean_common_terms
 3            0.060385                58               613                176.0        0.382001          0.387509       0.152782       0.614605           542.3375
 8            0.035004                25               436                 65.0        0.395318          0.379658       0.294352       0.503064           543.6125
 9            0.042690                25               301                 72.0        0.360076          0.357238       0.294452       0.434958           544.0375
12            0.034685                23               231                 62.5        0.357562          0.355434       0.301668       0.420616           542.7750
 7            0.050559                25               436                 58.0        0.358770          0.350133       0.273995       0.456486           542.1125
11            0.045492

In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cluster_selection["k"], cluster_selection["stability_median"], marker="o", label="median stability")
ax.plot(cluster_selection["k"], cluster_selection["silhouette_mindist"], marker="o", label="silhouette")
ax.axvline(FINAL_K, linestyle="--", linewidth=1)
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Score")
ax.set_title("Hierarchical cluster selection")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_stability" / "cluster_selection_scores.png", dpi=220)
plt.close(fig)

In [13]:
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt

D = mindist_tiebroken.to_numpy(dtype=float)
condensed = squareform(D, checks=False)
Z_final = linkage(condensed, method=LINKAGE_METHOD)

labels_final = fcluster(Z_final, t=FINAL_K, criterion="maxclust")
labels_final = pd.Series(labels_final, index=mindist_tiebroken.index, name="cluster")

cluster_sizes_final = (
    labels_final
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="n_terms")
)

cluster_sizes_final.to_csv(
    OUTPUT_DIR / "04_stability" / "cluster_sizes_final.csv",
    index=False,
)

plt.figure(figsize=(14, 6))
dendrogram(
    Z_final,
    no_labels=True,
    color_threshold=Z_final[-(FINAL_K - 1), 2],
    above_threshold_color="gray",
)
plt.title(f"Hierarchical clustering dendrogram, k = {FINAL_K}, linkage = {LINKAGE_METHOD}")
plt.xlabel("Search terms")
plt.ylabel("SAX MINDIST (tie-broken)")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "04_stability" / f"dendrogram_k{FINAL_K}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

cluster_sizes_final

,cluster,n_terms
0,1,176
1,2,58
2,3,613


## 5. Term-level stability, cluster summaries

In [ ]:
# -----------------------------------------------------------------------------
# Interpretable cluster summaries
# Inputs already defined:
#   mindist, mindist_tiebroken, panel_norm, stages, OUTPUT_DIR, LINKAGE_METHOD
#   CORE_STABILITY_QUANTILE, MIN_CORE_TERMS, TOP_N_CORE_TERMS (Section 0 config)
# -----------------------------------------------------------------------------
from pathlib import Path
import numpy as np
import pandas as pd

FINAL_K = 10   # <-- change this to re-run the whole chunk at a different k

# Recompute labels_final for the chosen k and the currently configured linkage
# method (passed explicitly so this isn't affected by any stale function-default
# binding from earlier in the session).
labels_final = hierarchical_labels(mindist_tiebroken, FINAL_K, method=LINKAGE_METHOD)

INDEX_DIR = OUTPUT_DIR / f"05_cluster_indices_{FINAL_K}"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

def term_stability_from_distance(
    full_distance: pd.DataFrame,
    final_labels: pd.Series,
) -> pd.DataFrame:
    rows = []

    for term in final_labels.index:
        cl = final_labels.loc[term]

        same_terms = final_labels.index[
            (final_labels == cl) & (final_labels.index != term)
        ]

        other_terms = final_labels.index[final_labels != cl]

        intra_dist = (
            full_distance.loc[term, same_terms].mean()
            if len(same_terms) > 0
            else np.nan
        )

        between_dist = (
            full_distance.loc[term, other_terms].mean()
            if len(other_terms) > 0
            else np.nan
        )

        margin = (
            between_dist - intra_dist
            if pd.notna(intra_dist) and pd.notna(between_dist)
            else np.nan
        )

        rows.append({
            "term": term,
            "cluster": int(cl),
            "mean_intra_mindist": intra_dist,
            "mean_between_mindist": between_dist,
            "mindist_margin": margin,
        })

    df = pd.DataFrame(rows)

    df["term_stability"] = (
        df.groupby("cluster")["mindist_margin"]
        .rank(pct=True, method="average")
    )

    return df.sort_values(
        ["cluster", "term_stability"],
        ascending=[True, False],
    )


def select_core_terms(
    term_stability: pd.DataFrame,
    top_n: int = TOP_N_CORE_TERMS,
) -> pd.DataFrame:
    rows = []

    for cl, g in term_stability.groupby("cluster"):
        g = g.sort_values(
            ["term_stability", "mindist_margin"],
            ascending=[False, False],
        ).copy()

        q = g["term_stability"].quantile(CORE_STABILITY_QUANTILE)
        core = g[g["term_stability"] >= q].copy()

        min_keep = min(MIN_CORE_TERMS, len(g))
        if len(core) < min_keep:
            core = g.head(min_keep).copy()

        core["core_rank"] = np.arange(1, len(core) + 1)
        rows.append(core)

    return pd.concat(rows, ignore_index=True)


def weighted_average(panel: pd.DataFrame, weights: pd.Series) -> pd.Series:
    cols = [c for c in weights.index if c in panel.columns]

    if len(cols) == 0:
        return pd.Series(np.nan, index=panel.index)

    w = weights.loc[cols].astype(float).fillna(0.0)

    if w.sum() <= 0:
        w = pd.Series(1 / len(cols), index=cols)
    else:
        w = w / w.sum()

    return panel[cols].mul(w, axis=1).sum(axis=1)


# -----------------------------------------------------------------------------
# Run summaries
# -----------------------------------------------------------------------------

term_stability = term_stability_from_distance(mindist, labels_final)
core_terms = select_core_terms(term_stability, top_n=TOP_N_CORE_TERMS)

term_stability.to_csv(INDEX_DIR / "term_stability.csv", index=False)
core_terms.to_csv(INDEX_DIR / "core_terms.csv", index=False)


# -----------------------------------------------------------------------------
# Cluster-level summary
# -----------------------------------------------------------------------------

cluster_summary = (
    term_stability
    .groupby("cluster")
    .agg(
        n_terms=("term", "size"),
        median_term_stability=("term_stability", "median"),
        mean_intra_mindist=("mean_intra_mindist", "mean"),
        mean_between_mindist=("mean_between_mindist", "mean"),
        mean_mindist_margin=("mindist_margin", "mean"),
    )
    .reset_index()
)

top_terms = (
    core_terms
    .sort_values(["cluster", "core_rank"])
    .groupby("cluster")["term"]
    .apply(lambda x: ", ".join(x.head(TOP_N_CORE_TERMS)))
)

cluster_summary["top_10_core_terms"] = (
    cluster_summary["cluster"]
    .map(top_terms)
)

cluster_summary.to_csv(INDEX_DIR / "cluster_summary.csv", index=False)

print(cluster_summary.to_string(index=False))

 cluster  n_terms  median_term_stability  mean_intra_mindist  mean_between_mindist  mean_mindist_margin                                                                                                                                                                                        top_10_core_terms
       1       90               0.505556           20.109549             31.097233            10.987685                                                                                                    2048, unblocked games, slope, quizizz, kahoot, prodigy, desmos, quizlet live, math playground, clever
       2       46               0.510870           24.834745             33.195747             8.361002                                   us election, election, election map, election news, presidential election, presidential results, presidential election results, poll results, elections, election live
       3       40               0.512500           28.528931             34.265781   

In [23]:
# -----------------------------------------------------------------------------
# Targeted re-clustering of a residual / heterogeneous cluster
# Inputs already defined: mindist_tiebroken, labels_final, LINKAGE_METHOD,
#   FINAL_K, hierarchical_labels, validation_for_k, stability_for_k,
#   term_stability_from_distance, select_core_terms (all from Sections 3-5)
# -----------------------------------------------------------------------------
RESIDUAL_CLUSTER_ID = 10          # <-- set to the cluster you want to re-split
RESIDUAL_K_RANGE = range(2, 11)
RESIDUAL_MIN_ACCEPTABLE_CLUSTER_SIZE = 8

residual_terms = labels_final[labels_final == RESIDUAL_CLUSTER_ID].index.to_list()
if len(residual_terms) < 4:
    raise ValueError(
        f"Cluster {RESIDUAL_CLUSTER_ID} has only {len(residual_terms)} terms -- "
        "too small to meaningfully re-cluster. Pick a different RESIDUAL_CLUSTER_ID."
    )

residual_mindist = mindist_tiebroken.loc[residual_terms, residual_terms]

RESIDUAL_DIR = OUTPUT_DIR / f"05_cluster_indices_{FINAL_K}" / f"residual_resplit_cluster_{RESIDUAL_CLUSTER_ID}"
RESIDUAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Re-clustering cluster {RESIDUAL_CLUSTER_ID}: {len(residual_terms)} terms, linkage={LINKAGE_METHOD}")

# --- sub-k sweep, same machinery as the global k-selection in Section 4 ---
residual_validation_rows = []
residual_stability_tables = []

for k in RESIDUAL_K_RANGE:
    if k >= len(residual_terms):
        continue
    labels_k = hierarchical_labels(residual_mindist, k, method=LINKAGE_METHOD)
    residual_validation_rows.append(validation_for_k(residual_mindist, labels_k, k))
    residual_stability_tables.append(
        stability_for_k(
            residual_mindist,
            k=k,
            n_subsamples=STABILITY_SUBSAMPLES,
            fraction=STABILITY_FRACTION,
            seed=RANDOM_STATE + 1000 + k,
            method=LINKAGE_METHOD,
        )
    )

residual_validation_df = pd.DataFrame(residual_validation_rows)
residual_stability_raw = pd.concat(residual_stability_tables, ignore_index=True)
residual_stability_summary = (
    residual_stability_raw
    .groupby("k")
    .agg(
        stability_mean=("ari", "mean"),
        stability_median=("ari", "median"),
        stability_p10=("ari", lambda x: x.quantile(0.10)),
        stability_p90=("ari", lambda x: x.quantile(0.90)),
        mean_common_terms=("n_common", "mean"),
    )
    .reset_index()
)

residual_selection = residual_validation_df.merge(residual_stability_summary, on="k", how="left")
residual_selection.to_csv(RESIDUAL_DIR / "residual_cluster_selection_by_k.csv", index=False)
residual_stability_raw.to_csv(RESIDUAL_DIR / "residual_subsample_stability_raw.csv", index=False)

print(residual_selection.sort_values(["stability_median", "silhouette_mindist"], ascending=False).to_string(index=False))

# --- pick sub-k the same way FINAL_K was picked: stability + silhouette, floor on min size ---
def select_k_from_stability_local(selection_df: pd.DataFrame, min_cluster_size: int) -> int:
    eligible = selection_df[selection_df["min_cluster_size"] >= min_cluster_size]
    if eligible.empty:
        eligible = selection_df
    best = eligible.sort_values(["stability_median", "silhouette_mindist"], ascending=False).iloc[0]
    return int(best["k"])


residual_k = select_k_from_stability_local(residual_selection, RESIDUAL_MIN_ACCEPTABLE_CLUSTER_SIZE)
print(f"\nSelected sub-k for residual re-split: {residual_k}")

residual_labels = hierarchical_labels(residual_mindist, residual_k, method=LINKAGE_METHOD)
residual_labels.rename_axis("term").reset_index().to_csv(
    RESIDUAL_DIR / "residual_cluster_assignments.csv", index=False
)

# --- reuse the same per-term stability + core-term + summary machinery from Section 5 ---
residual_term_stability = term_stability_from_distance(residual_mindist, residual_labels)
residual_core_terms = select_core_terms(residual_term_stability, top_n=TOP_N_CORE_TERMS)
residual_term_stability.to_csv(RESIDUAL_DIR / "residual_term_stability.csv", index=False)
residual_core_terms.to_csv(RESIDUAL_DIR / "residual_core_terms.csv", index=False)

residual_summary = (
    residual_term_stability
    .groupby("cluster")
    .agg(
        n_terms=("term", "size"),
        median_term_stability=("term_stability", "median"),
        mean_intra_mindist=("mean_intra_mindist", "mean"),
        mean_between_mindist=("mean_between_mindist", "mean"),
        mean_mindist_margin=("mindist_margin", "mean"),
    )
    .reset_index()
)
residual_top_terms = (
    residual_core_terms
    .sort_values(["cluster", "core_rank"])
    .groupby("cluster")["term"]
    .apply(lambda x: ", ".join(x.head(TOP_N_CORE_TERMS)))
)
residual_summary["top_10_core_terms"] = residual_summary["cluster"].map(residual_top_terms)
residual_summary.to_csv(RESIDUAL_DIR / "residual_cluster_summary.csv", index=False)

# --- small dendrogram of just this residual cluster, for a visual sanity check ---
Z_residual = linkage(squareform(residual_mindist.to_numpy(dtype=float), checks=False), method=LINKAGE_METHOD)
plt.figure(figsize=(12, 5))
dendrogram(
    Z_residual,
    no_labels=True,
    color_threshold=Z_residual[-(residual_k - 1), 2] if residual_k > 1 else None,
    above_threshold_color="gray",
)
plt.title(f"Residual re-split of cluster {RESIDUAL_CLUSTER_ID}, sub-k = {residual_k}, linkage = {LINKAGE_METHOD}")
plt.xlabel("Terms")
plt.ylabel("SAX MINDIST (tie-broken)")
plt.tight_layout()
plt.savefig(RESIDUAL_DIR / f"residual_dendrogram_subk{residual_k}.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\nInterpretation guide: compare each mean_mindist_margin below to the parent")
print(f"cluster {RESIDUAL_CLUSTER_ID}'s own margin in cluster_summary.csv. Sub-clusters with")
print("noticeably higher margins than the parent indicate real recovered structure;")
print("sub-clusters with margins as weak as (or weaker than) the parent suggest this")
print("portion of the panel may not separate further under this distance metric.")
print()
print(residual_summary.to_string(index=False))

Re-clustering cluster 10: 88 terms, linkage=ward
 k  silhouette_mindist  min_cluster_size  max_cluster_size  median_cluster_size  stability_mean  stability_median  stability_p10  stability_p90  mean_common_terms
 2            0.059936                26                62                 44.0        0.574332          0.611881       0.222655       0.910891            55.5250
 9            0.100306                 4                14                  9.0        0.603296          0.594677       0.467944       0.752409            55.8625
 8            0.102471                 4                23                  8.5        0.581078          0.576169       0.458076       0.706870            55.7375
 7            0.089550                 8                23                 12.0        0.558019          0.574982       0.383312       0.696491            55.8250
10            0.110767                 3                14                  8.5        0.584066          0.565730       0.473434       0

## 06. Validity checks

In [26]:
from scipy import stats as _stats

standardized = (paa_flat - paa_flat.mean()) / paa_flat.std()
excess_kurtosis = _stats.kurtosis(paa_flat, fisher=True)   # 0 for a true Gaussian
ks_stat, ks_p = _stats.kstest(standardized, "norm")

breakpoint_fit = pd.DataFrame({
    "metric": ["excess_kurtosis", "ks_statistic", "ks_pvalue", "n_paa_coefficients", "breakpoint_mode_used"],
    "value": [excess_kurtosis, ks_stat, ks_p, len(paa_flat), BREAKPOINT_MODE],
})
breakpoint_fit.to_csv(OUTPUT_DIR / "06_validity_checks" / "paa_gaussian_fit_diagnostic.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(paa_flat, bins=60, density=True, alpha=0.7, label="pooled PAA coefficients")
xs = np.linspace(paa_flat.min(), paa_flat.max(), 200)
axes[0].plot(xs, norm.pdf(xs, paa_flat.mean(), paa_flat.std()), lw=2, label="fitted Gaussian")
axes[0].set_title("Pooled PAA coefficients vs Gaussian fit")
axes[0].legend(frameon=False)
_stats.probplot(paa_flat, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot vs Gaussian")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_validity_checks" / "paa_gaussian_fit_diagnostic.png", dpi=200)
plt.close(fig)

print(f"Excess kurtosis: {excess_kurtosis:.3f}  (0 = Gaussian; > 0 = more peaked than Gaussian)")
print(f"KS test vs N(0,1): stat={ks_stat:.4f}, p={ks_p:.2e}")
print(f"-> breakpoint_mode currently in use: {BREAKPOINT_MODE}")

Excess kurtosis: 3.887  (0 = Gaussian; > 0 = more peaked than Gaussian)
KS test vs N(0,1): stat=0.1457, p=0.00e+00
-> breakpoint_mode currently in use: empirical


In [27]:
def mindist_tightness_check(
    panel: pd.DataFrame,
    segment_grid: list[int],
    alphabet_grid: list[int],
    sample_size: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    cols = panel.columns.to_list()
    sample_cols = rng.choice(cols, size=min(sample_size, len(cols)), replace=False)
    sub = panel[sample_cols]

    X = panel_to_tslearn_array(sub)
    n_timestamps = sub.shape[0]
    flat = X[:, :, 0]
    euclid = squareform(pdist(flat, metric="euclidean"))
    mask = ~np.eye(len(sample_cols), dtype=bool)

    rows = []
    for w in segment_grid:
        paa_w = PiecewiseAggregateApproximation(n_segments=w).fit_transform(X)[:, :, 0]
        for a in alphabet_grid:
            bp = (
                empirical_breakpoints(paa_w.ravel(), a)
                if BREAKPOINT_MODE == "empirical"
                else sax_breakpoints(a)
            )
            codes = discretize_paa(paa_w, bp)
            sd = sax_symbol_distance_table(a, bp)
            scale = np.sqrt(n_timestamps / w)
            per_seg = sd[codes[:, None, :], codes[None, :, :]]
            md = scale * np.sqrt((per_seg ** 2).sum(axis=2))

            ratio = md[mask] / np.clip(euclid[mask], 1e-9, None)
            rows.append({
                "n_segments": w,
                "alphabet_size": a,
                "mean_tightness_ratio": float(np.mean(ratio)),
                "median_tightness_ratio": float(np.median(ratio)),
                "share_mindist_exceeds_euclid": float(np.mean(md[mask] > euclid[mask] + 1e-6)),
            })
    return pd.DataFrame(rows)


tightness_df = mindist_tightness_check(
    panel_sax, SEGMENT_GRID, ALPHABET_GRID, ABLATION_SAMPLE_SIZE, RANDOM_STATE
)
tightness_df.to_csv(OUTPUT_DIR / "06_validity_checks" / "mindist_tightness_by_w_a.csv", index=False)

chosen_row = tightness_df[
    (tightness_df["n_segments"] == N_SEGMENTS) & (tightness_df["alphabet_size"] == ALPHABET_SIZE)
]
print("(w, a) tightness ablation (higher mean_tightness_ratio = less information lost):")
print(tightness_df.sort_values("mean_tightness_ratio", ascending=False).to_string(index=False))
print("\nCurrently configured (N_SEGMENTS, ALPHABET_SIZE) row:")
print(chosen_row.to_string(index=False))

(w, a) tightness ablation (higher mean_tightness_ratio = less information lost):
 n_segments  alphabet_size  mean_tightness_ratio  median_tightness_ratio  share_mindist_exceeds_euclid
        128             11              0.388816                0.391334                           0.0
         96             11              0.360789                0.361860                           0.0
        128              9              0.348101                0.350331                           0.0
         96              9              0.322465                0.323694                           0.0
         64             11              0.303398                0.303838                           0.0
        128              7              0.292205                0.293450                           0.0
         64              9              0.273833                0.274399                           0.0
         96              7              0.273356                0.273721                       

In [28]:
_k_for_linkage_check = FINAL_K if FINAL_K is not None else int(
    cluster_selection.sort_values(["stability_median", "silhouette_mindist"], ascending=False).iloc[0]["k"]
)

linkage_rows = []
for method in LINKAGE_CANDIDATES:
    labels_m = hierarchical_labels(mindist_tiebroken, _k_for_linkage_check, method=method)
    val = validation_for_k(mindist_tiebroken, labels_m, _k_for_linkage_check)
    stab = stability_for_k(
        mindist_tiebroken, _k_for_linkage_check, STABILITY_SUBSAMPLES, STABILITY_FRACTION, RANDOM_STATE, method=method
    )
    linkage_rows.append({
        "linkage_method": method,
        "k_used": _k_for_linkage_check,
        "silhouette_mindist": val["silhouette_mindist"],
        "min_cluster_size": val["min_cluster_size"],
        "max_cluster_size": val["max_cluster_size"],
        "stability_mean": stab["ari"].mean(),
        "stability_median": stab["ari"].median(),
    })

linkage_comparison = pd.DataFrame(linkage_rows)
linkage_comparison.to_csv(OUTPUT_DIR / "06_validity_checks" / "linkage_method_comparison.csv", index=False)
print(f"\nLinkage method comparison at k={_k_for_linkage_check} (currently configured LINKAGE_METHOD = '{LINKAGE_METHOD}'):")
print(linkage_comparison.sort_values("stability_median", ascending=False).to_string(index=False))

def select_k_from_stability(cluster_selection: pd.DataFrame, min_cluster_size: int) -> int:
    eligible = cluster_selection[cluster_selection["min_cluster_size"] >= min_cluster_size]
    if eligible.empty:
        eligible = cluster_selection
    best = eligible.sort_values(["stability_median", "silhouette_mindist"], ascending=False).iloc[0]
    return int(best["k"])


auto_k = select_k_from_stability(cluster_selection, MIN_ACCEPTABLE_CLUSTER_SIZE)
manual_k = FINAL_K
EFFECTIVE_K = auto_k if AUTO_SELECT_K else manual_k

k_selection_log = pd.DataFrame({
    "metric": ["auto_selected_k", "manual_final_k", "min_acceptable_cluster_size", "auto_select_k_enabled", "effective_k_used"],
    "value": [auto_k, manual_k, MIN_ACCEPTABLE_CLUSTER_SIZE, AUTO_SELECT_K, EFFECTIVE_K],
})
k_selection_log.to_csv(OUTPUT_DIR / "06_validity_checks" / "k_selection_log.csv", index=False)

FINAL_K = EFFECTIVE_K
print(f"\nData-driven k (stability + silhouette, min cluster size >= {MIN_ACCEPTABLE_CLUSTER_SIZE}): {auto_k}")
print(f"Manually configured FINAL_K: {manual_k}   ->   AUTO_SELECT_K={AUTO_SELECT_K}   ->   effective k: {FINAL_K}")


Linkage method comparison at k=6 (currently configured LINKAGE_METHOD = 'ward'):
linkage_method  k_used  silhouette_mindist  min_cluster_size  max_cluster_size  stability_mean  stability_median
        single       6            0.087703                 1               841        0.749206          0.747460
       average       6            0.097275                 1               834        0.723652          0.735583
          ward       6            0.042784                25               436        0.361022          0.348993
      complete       6            0.039568                28               284        0.169125          0.155883

Data-driven k (stability + silhouette, min cluster size >= 15): 3
Manually configured FINAL_K: 6   ->   AUTO_SELECT_K=True   ->   effective k: 3


## 07. Poster

In [ ]:
# -----------------------------------------------------------------------------
# Shared periodicity diagnostic
# Distinguishes "genuinely periodic" from "periodogram picked a peak out of an
# otherwise flat/noisy spectrum" -- the bug behind clusters 1, 6, 9 collapsing
# to a ~2-week display window, and the reason clusters 1/2/5/7/10 show comet
# shapes rather than loops in the phase portraits (which is the CORRECT
# behavior for event-driven series, but needs to be labeled as such rather
# than silently treated the same as real periodicity).
# -----------------------------------------------------------------------------
from scipy.signal import periodogram

def periodicity_diagnostics(
    series: np.ndarray,
    min_period: int = 3,
    max_period: int = 730,
    significance_ratio: float = 3.0,
) -> tuple[int, bool, float]:
    """
    Returns (dominant_period, is_significant, concentration).

    concentration = power at the dominant period / mean power across the
    searched band. is_significant = concentration >= significance_ratio,
    i.e. the peak clearly stands out from a roughly flat (noise-like)
    spectrum rather than being an arbitrary highest point in the noise floor.
    """
    x = series - np.nanmean(series)
    freqs, power = periodogram(x, fs=1.0)
    periods = np.zeros_like(freqs)
    valid = freqs > 0
    periods[valid] = 1.0 / freqs[valid]
    mask = (periods >= min_period) & (periods <= max_period)

    if not mask.any() or power[mask].sum() == 0:
        return min_period, False, 0.0

    band_power = power[mask]
    band_periods = periods[mask]
    peak_idx = np.argmax(band_power)
    peak_period = int(round(band_periods[peak_idx]))
    concentration = float(band_power[peak_idx] / (band_power.mean() + 1e-12))
    is_significant = concentration >= significance_ratio

    return peak_period, is_significant, concentration


def dominant_period(series: np.ndarray, min_period: int = 3, max_period: int = 730) -> int:
    """Back-compat thin wrapper (period only, no significance check)."""
    period, _, _ = periodicity_diagnostics(series, min_period, max_period)
    return period


def auto_smooth_window(series: np.ndarray, max_lag: int = 60) -> int:
    """
    Smoothing window from the series' own ACF decay: the lag where
    autocorrelation first drops inside the +/-1.96/sqrt(n) noise band.
    This is deliberately a SEPARATE quantity from the display window below --
    conflating the two in a single 'window=Xd' label was the source of the
    confusing archetype titles.
    """
    x = pd.Series(series).interpolate().bfill().ffill().to_numpy()
    n = len(x)
    a = acf(x, nlags=min(max_lag, n // 3), fft=True)
    threshold = 1.96 / np.sqrt(n)
    below = np.where(np.abs(a[1:]) < threshold)[0]
    window = int(below[0] + 1) if len(below) else 7
    return max(3, window | 1)


from statsmodels.tsa.stattools import acf


def select_display_window(
    panel: pd.DataFrame,
    terms: list[str],
    cycles: float = 2.5,
    significance_ratio: float = 3.0,
    min_significant_share: float = 0.5,
) -> tuple[str, str, str]:
    """
    Show `cycles` periods of the cluster's dominant period IF a majority of
    its member terms actually have a significant periodic peak. Otherwise,
    fall back to the FULL available range -- shrinking the window based on a
    meaningless periodogram peak (as happened for clusters 1, 6, 9) actively
    hides the true pattern instead of revealing it.

    Returns (start, end, basis) where `basis` is a short string documenting
    which branch was taken, for direct inclusion in the plot title.
    """
    diags = [
        periodicity_diagnostics(panel[t].to_numpy(dtype=float), significance_ratio=significance_ratio)
        for t in terms if t in panel.columns
    ]
    end = panel.index.max()

    if not diags:
        return panel.index.min().strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), "full range (no terms)"

    sig_flags = [d[1] for d in diags]
    frac_significant = float(np.mean(sig_flags))

    if frac_significant >= min_significant_share:
        sig_periods = [d[0] for d in diags if d[1]]
        period = int(np.median(sig_periods))
        start = max(end - pd.Timedelta(days=int(period * cycles)), panel.index.min())
        basis = f"periodic, period={period}d ({frac_significant:.0%} of terms significant)"
    else:
        start = panel.index.min()
        basis = f"full range (only {frac_significant:.0%} of terms show significant periodicity)"

    return start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), basis

# -----------------------------------------------------------------------------
# Cluster archetype panel: distinct-shape small multiples
# One subplot per cluster: centroid trajectory + IQR band, each on its own
# auto-selected window/smoothing, so genuinely different dynamic ranges and
# cycle lengths stay visible instead of being normalized away.
# -----------------------------------------------------------------------------

POSTER_DIR = OUTPUT_DIR / "08_poster"
ARCHETYPE_DIR = POSTER_DIR / "cluster_archetypes"
ARCHETYPE_DIR.mkdir(parents=True, exist_ok=True)

clusters_sorted = sorted(labels_final.unique())
ncols = 3
nrows = int(np.ceil(len(clusters_sorted) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.3 * nrows), squeeze=False)

archetype_meta = []
for i, cl in enumerate(clusters_sorted):
    ax = axes[i // ncols, i % ncols]
    members = labels_final[labels_final == cl].index.intersection(panel_norm.columns).to_list()

    display_start, display_end, basis = select_display_window(panel_norm, members, cycles=2.5)
    sub = panel_norm.loc[display_start:display_end, members]

    smooth_window = auto_smooth_window(sub.mean(axis=1).to_numpy())
    smoothed = sub.rolling(smooth_window, center=True, min_periods=1).mean()

    centroid = smoothed.mean(axis=1)
    q25, q75 = smoothed.quantile(0.25, axis=1), smoothed.quantile(0.75, axis=1)

    ax.fill_between(smoothed.index, q25, q75, alpha=0.25, linewidth=0)
    ax.plot(smoothed.index, centroid, linewidth=2.0)

    # Two separate, clearly-labeled quantities: which dates are shown (basis)
    # vs. how much the display trace is smoothed (smooth_window).
    ax.set_title(f"Cluster {cl}  (n={len(members)}, smooth={smooth_window}d)\n{basis}", fontsize=8.5)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    archetype_meta.append({
        "cluster": int(cl), "n_terms": len(members),
        "display_start": display_start, "display_end": display_end,
        "display_window_basis": basis,
        "smooth_window_days": smooth_window,
    })

for j in range(len(clusters_sorted), nrows * ncols):
    axes[j // ncols, j % ncols].axis("off")

fig.suptitle("Cluster archetypes: centroid \u00b1 IQR (display window and smoothing window are independent)", fontsize=12.5)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(ARCHETYPE_DIR / "cluster_archetype_panel.png", dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(archetype_meta).to_csv(ARCHETYPE_DIR / "cluster_archetype_window_selection.csv", index=False)

In [30]:
from statsmodels.tsa.seasonal import STL

def shape_signature(series: pd.Series, period: int) -> dict:
    """
    A small feature vector describing a series' shape -- trend strength,
    seasonal strength, spikiness, dominant period -- independent of SAX
    strings, so new terms can be compared to existing cluster archetypes
    directly, without rebuilding the MINDIST matrix.
    """
    x = series.interpolate().bfill().ffill()
    try:
        res = STL(x, period=max(3, period), robust=True).fit()
        var_resid = np.var(res.resid)
        trend_strength = max(0.0, 1 - var_resid / np.var(res.trend + res.resid))
        seasonal_strength = max(0.0, 1 - var_resid / np.var(res.seasonal + res.resid))
    except Exception:
        trend_strength, seasonal_strength = np.nan, np.nan

    return {
        "trend_strength": trend_strength,
        "seasonal_strength": seasonal_strength,
        "spikiness_kurtosis": float(pd.Series(x).kurt()),
        "dominant_period_days": dominant_period(x.to_numpy()),
    }


cluster_dictionary = []
for cl in clusters_sorted:
    members = labels_final[labels_final == cl].index.intersection(panel_norm.columns).to_list()
    centroid = panel_norm[members].mean(axis=1)
    sig = shape_signature(centroid, period=dominant_period(centroid.to_numpy()))
    sig.update({"cluster": int(cl), "n_terms": len(members)})
    cluster_dictionary.append(sig)

cluster_dictionary_df = pd.DataFrame(cluster_dictionary).set_index("cluster")
cluster_dictionary_df.to_csv(POSTER_DIR / "cluster_shape_dictionary.csv")


def classify_new_term(new_series: pd.Series, dictionary: pd.DataFrame) -> pd.Series:
    """Rank existing cluster archetypes by similarity to a brand-new term's shape."""
    cols = ["trend_strength", "seasonal_strength", "spikiness_kurtosis", "dominant_period_days"]
    sig = pd.Series(shape_signature(new_series, dominant_period(new_series.to_numpy())))[cols]
    ref = dictionary[cols]
    ref_std = (ref - ref.mean()) / ref.std().replace(0, 1)
    sig_std = (sig - ref.mean()) / ref.std().replace(0, 1)
    return np.sqrt(((ref_std - sig_std) ** 2).sum(axis=1)).sort_values()

In [31]:
# -----------------------------------------------------------------------------
# Delay-embedding phase-portrait panel
# One subplot per cluster: x(t) vs x(t - lag), lag = the cluster's own
# auto-selected dominant period. A closed loop indicates a periodic shape;
# a diagonal streak indicates a trending shape; a shapeless cloud indicates
# noise-dominated behavior -- a more literal geometric analogue of shape
# than the raw time-series trace itself.
# -----------------------------------------------------------------------------
from scipy.signal import periodogram

PHASE_DIR = OUTPUT_DIR / "08_poster" / "phase_portraits"
PHASE_DIR.mkdir(parents=True, exist_ok=True)


def dominant_period(series: np.ndarray, min_period: int = 3, max_period: int = 730) -> int:
    """
    The lag with the most spectral power in the series' periodogram -- used
    here as the embedding lag, so each cluster's phase portrait is built at
    its own natural cycle length rather than one fixed lag for everything.
    """
    x = series - np.nanmean(series)
    freqs, power = periodogram(x, fs=1.0)
    periods = np.zeros_like(freqs)
    valid = freqs > 0
    periods[valid] = 1.0 / freqs[valid]
    mask = (periods >= min_period) & (periods <= max_period)
    if not mask.any():
        return min_period
    return int(round(periods[mask][np.argmax(power[mask])]))


def phase_embed(x: np.ndarray, lag: int) -> tuple[np.ndarray, np.ndarray]:
    """Standard delay embedding: pairs (x[t], x[t - lag]) for t >= lag."""
    if lag < 1 or lag >= len(x):
        lag = max(1, len(x) // 4)
    return x[lag:], x[:-lag]


clusters_sorted = sorted(labels_final.unique())
ncols = 3
nrows = int(np.ceil(len(clusters_sorted) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.2 * nrows), squeeze=False)

phase_meta = []
for i, cl in enumerate(clusters_sorted):
    ax = axes[i // ncols, i % ncols]
    members = labels_final[labels_final == cl].index.intersection(panel_norm.columns).to_list()

    centroid = panel_norm[members].mean(axis=1).interpolate().bfill().ffill().to_numpy()
    lag = dominant_period(centroid)

    x_t, x_lag = phase_embed(centroid, lag)

    # color by time so the trajectory's direction/progression is visible, not just its shape
    t_idx = np.arange(len(x_t))
    sc = ax.scatter(x_lag, x_t, c=t_idx, cmap="viridis", s=6, alpha=0.7, linewidths=0)
    ax.plot(x_lag, x_t, color="gray", alpha=0.15, linewidth=0.6)

    ax.set_title(f"Cluster {cl}  (n={len(members)}, lag={lag}d)", fontsize=10)
    ax.set_xlabel(f"x(t - {lag})", fontsize=8)
    ax.set_ylabel("x(t)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_aspect("equal", adjustable="box")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    phase_meta.append({"cluster": int(cl), "n_terms": len(members), "embedding_lag_days": lag})

for j in range(len(clusters_sorted), nrows * ncols):
    axes[j // ncols, j % ncols].axis("off")

fig.suptitle("Cluster phase portraits: delay embedding at each cluster's dominant period", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(PHASE_DIR / "cluster_phase_portrait_panel.png", dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(phase_meta).to_csv(PHASE_DIR / "phase_portrait_lag_selection.csv", index=False)